# Learned-local V2

**Trạng thái:** `FROZEN_SOURCE_AND_SMOKE_REPORTS`  
**Chế độ mặc định:** CPU · read-only · không train · không resume · không chạy test inference.  

Cải tiến Learned-local: anchor-uniform, k=3 neighbors cùng lớp, rho=0.2; source kiểm kỹ thuật được khóa riêng.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'manifest.json').is_file() and (ROOT.parent / 'manifest.json').is_file():
    ROOT = ROOT.parent
elif not (ROOT / 'manifest.json').is_file() and (ROOT / 'KLTN_clean' / 'manifest.json').is_file():
    ROOT = ROOT / 'KLTN_clean'
if not (ROOT / 'manifest.json').is_file():
    raise FileNotFoundError('Project root not found. Open the extracted project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.audit import (verify_sources, scan, zip_listing, zip_job_contracts,
                       read_test_summary, read_test_by_seed, npz_headers,
                       source_functions, function_source, statuses, export_json)

# Optional directories containing original ZIPs and data.
import json
from src.audit import resolve_inputs
INPUT_CONFIG = resolve_inputs(ROOT)
INPUTS = INPUT_CONFIG['directories']
from src.results import (read_csv_file_or_zip, metrics_by_seed,
                         test_paired_deltas, recompute_summary,
                         check_against_stored, weighting_tables)
print('Project:', ROOT)

Project: C:\Users\ADMIN\OneDrive\Tài liệu\ChatGPT\KLTN\KLTN_clean


## Source lock

In [2]:
checks = verify_sources(ROOT)
print('Frozen source files:', len(checks), '| PASS:', sum(x['status']=='PASS' for x in checks))

Frozen source files: 15 | PASS: 15


## Thiết kế

Cấu trúc generator/head; không train, không tạo TensorFlow objects.

In [3]:
m4=ROOT/'src/frozen/m4.py'
methods=function_source(m4,['build_graph_per_class','tao_head_v2','infer_fake'])
for name, src in methods.items():print(name,'|',len(src.splitlines()),'lines')

build_graph_per_class | 25 lines
tao_head_v2 | 31 lines
infer_fake | 52 lines


## Kiểm kỹ thuật

Đã lưu source smoke, matched runner và technical check. Trạng thái thực thi phải dựa trên ZIP report.

In [4]:
from src.audit import digest
for n in ('smoke.py','runner.py','technical.py'):
    p=ROOT/'src/frozen'/n
    print(n, digest(p)[:16], 'functions:',len(source_functions(p)))

smoke.py dfdb2c59d4905c15 functions: 25
runner.py 3ac45bc3d2290590 functions: 48
technical.py 1024fd19899d0ebe functions: 55


## Báo cáo

Mốc 3 chưa tìm thấy report; không coi kiểm tra đã PASS.

In [5]:
reports=statuses([ROOT/'outputs',*INPUTS],['MOC2_IP_SMOKE','MOC3_IP_FINAL'])
for key,value in reports.items():
    print(key, value if value else 'MISSING')

MOC2_IP_SMOKE ['C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\outputs\\technical\\MOC2_IP_SMOKE_20261015T133956Z_2abba22f_REPORT.zip']
MOC3_IP_FINAL MISSING


## Giới hạn

Không thay đổi source huấn luyện đã chạy, không mở test mới và không kết luận PASS khi thiếu output gốc. Kết quả chỉ có giá trị trong phạm vi protocol B của Indian Pines.

In [6]:
import sys
assert 'tensorflow' not in sys.modules, 'Unexpected TensorFlow/model initialization'
print('CPU_ONLY_NO_MODEL_IMPORT_PASS')

CPU_ONLY_NO_MODEL_IMPORT_PASS
